# 5. Repensando el Overfitting

#### 5.1 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 5.2 rpart  Canaritos

Se agregarán canaritos al dataset, se entrenará el arbol con los mejores hiperparámetros encontrados, y se analizará si los canaritos aparecen en algun split.

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

### 5.2.1  carga manual de hiperparámetros
Aqui debe cargar SU semilla primigenia y
<br> SUS mejores hiperparámetros que encontró para el ARBOL DE DECISION, ya sea por Grid Search o  Bayesian Optimization

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 600
PARAM$rpart$minbucket <- 150
PARAM$rpart$maxdepth <- 6

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5200"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semilla_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

vaya a su Google Drive **exp / exp5200**
<br> baje el archivo **arbol_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hechos en alguna de las nuevas variables canaritos



---



## 5.3 rpart  Canaritos desconfiados

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia y
<br> parametros de un@ alumn@ desconfiad@

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 2000
PARAM$rpart$minbucket <- 800
PARAM$rpart$maxdepth <- 6

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5300"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semilla_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos_desconfiados.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

vaya a su Google Drive
<br> busque la carpeta **/ exp / exp5300**
<br> baje el archivo **arbol_canaritos_desconfiados.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hecho en alguna de las nuevas variables canaritos



---



## 5.4 rpart  Canaritos pruning

Se trabaja con la original clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 1021917

PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5400"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semilla_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

In [ ]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_ternaria=="BAJA+2", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_ternaria ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [ ]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

vaya a su Google Drive
<br> busque la carpeta **exp / exp5400**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---



## 5.5 LightGBM canaritos en Feature Importance



Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
if(!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5500"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()

PARAM$semilla_primigenia <- 1021917

In [ ]:
# Parametros LightGBM
PARAM$lgbm <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,
  num_threads= 16,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,
  feature_fraction= 0.50,
  feature_fraction_bynode= 1.0,

  num_iterations= 52,
  learning_rate= 0.11,
  num_leaves= 294,
  min_sum_hessian_in_leaf= 48,
  min_gain_to_split= 6.2e-16
)


In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5500"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semilla_primigenia)

# agrego los siguientes canaritos
for( i in 1:60 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
 campos_buenos <- setdiff(colnames(dataset), "clase_ternaria")

In [ ]:
# construyo dataset en formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[ foto_mes==202107, campos_buenos, with= FALSE]),
  label= dataset[ foto_mes==202107,  ifelse( clase_ternaria=="CONTINUA", 0L, 1L ) ],
  free_raw_data= FALSE
)


In [ ]:
# entreno el modelo

modelo <- lgb.train(
  data= dtrain,
  param= PARAM$lgbm
)

In [ ]:
# genero la importancia de variables
tb_importancia <- lgb.importance(modelo)

In [ ]:
fwrite( tb_importancia,
  file= "tb_importancia.txt",
  sep= "\t"
)

In [ ]:
print(tb_importancia, nrows=500)

vaya a su Google Drive
<br> busque la carpeta **exp / exp5500**
<br> baje el archivo **tb_importancia.txt**  a su laptop





---

